In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import matplotlib.pyplot as plt
import os
import shutil
import tensorflow as tf
from tensorflow.keras import layers, models, applications

# # 1. 하이퍼파라미터 및 설정 (보고서 기반)
# IMG_WIDTH = 640
# IMG_HEIGHT = 480
# BATCH_SIZE = 32
# NUM_CLASSES = 3  # 정상, 경미, 중증도
# LEARNING_RATE = 0.001
증ㄱ
# # 원본 이미지 경로
# original_data_dir = '/content/drive/MyDrive/My_Dataset' # 여기에 Gemini로 생성한 10장씩 들어있는 폴더 경로
# output_augmented_dir = '/content/drive/MyDrive/Augmented_Dataset' # 증강된 이미지 저장할 경로

# # ImageDataGenerator 설정 (보고서 내용 기반: 좌우 반전, 밝기 조절)
# datagen = ImageDataGenerator(
#     rotation_range=0,          # 회전은 자세 왜곡될 수 있으니 0으로 유지
#     width_shift_range=0.0,     # 좌우 이동도 0으로 유지
#     height_shift_range=0.0,    # 상하 이동도 0으로 유지
#     horizontal_flip=True,      # 좌우 반전 (보고서 언급)
#     brightness_range=[0.8, 1.2], # 밝기 조절 (보고서 언급)
#     zoom_range=0.0,            # 확대/축소도 0으로 유지
#     fill_mode='nearest'
# )

# # 증강된 이미지 저장할 폴더 생성
# for class_name in ['Normal', 'Mild', 'Severe']:
#     os.makedirs(os.path.join(output_augmented_dir, class_name), exist_ok=True)

# # 클래스별로 증강 및 저장
# for class_name in ['Normal', 'Mild', 'Severe']:
#     class_path = os.path.join(original_data_dir, class_name)
#     output_path = os.path.join(output_augmented_dir, class_name)

#     # 원본 이미지 파일 리스트
#     image_files = [os.path.join(class_path, f) for f in os.listdir(class_path) if f.endswith(('.png', '.jpg', '.jpeg'))]

    # 각 원본 이미지에 대해 증강 수행 (5배 증강)
    for i, img_path in enumerate(image_files):
        img = tf.keras.utils.load_img(img_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
        x = tf.keras.utils.img_to_array(img)
        x = x.reshape((1,) + x.shape) # (1, 높이, 너비, 채널) 형태로 reshape

        # 원본 이미지도 포함 (총 1+4 = 5장)
        shutil.copy(img_path, os.path.join(output_path, f'original_{i}.jpg'))

        count = 0
        for batch in datagen.flow(x, batch_size=1,
                                  save_to_dir=output_path,
                                  save_prefix=f'aug_{i}',
                                  save_format='jpg'):
            count += 1
            if count >= 4: # 원본 포함 총 5장이 되도록 4장 증강
                break

# print(f"이미지 증강 완료. 증강된 데이터는 {output_augmented_dir} 에 저장되었습니다.")

## 이제 train_ds와 val_ds를 로드할 때 이 output_augmented_dir 경로를 사용하면 돼!


In [ ]:
data_dir = '/content/drive/MyDrive/Augmented_Dataset'

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import pathlib

# [중요] 증강된 데이터가 저장된 경로로 설정 (이전 단계에서 설정한 경로)
# 예: '/content/drive/MyDrive/Augmented_Dataset'
# dataset_path = '/content/drive/MyDrive/Augmented_Dataset'
# data_dir = pathlib.Path(dataset_path)

# 파라미터 설정 (보고서 기준: 640x480)
IMG_HEIGHT = 480
IMG_WIDTH = 640
BATCH_SIZE = 32

print("데이터 로드 중...")

# 1. 학습 데이터셋 (80%)
train_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(IMG_HEIGHT, IMG_WIDTH),
  batch_size=BATCH_SIZE
)

# 2. 검증 데이터셋 (20%)
val_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(IMG_HEIGHT, IMG_WIDTH),
  batch_size=BATCH_SIZE
)

# 3. 클래스 이름 확인 (이 순서가 나중에 앱에서 0, 1, 2의 의미가 됨)
class_names = train_ds.class_names
print(f"\n[중요] 라벨 순서 (0, 1, 2): {class_names}")
# 예: ['Mild', 'Normal', 'Severe'] 라면 -> 0: Mild, 1: Normal, 2: Severe

# 4. 성능 최적화 (Caching & Prefetching)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
# # 2. 데이터 전처리 및 증강 (Data Augmentation)
# # 보고서에 따르면 좌우 반전, 밝기 조절 등을 사용했어.
# data_augmentation = tf.keras.Sequential([
#     layers.Resizing(IMG_HEIGHT, IMG_WIDTH),
#     # layers.Rescaling(1./255), # 0~1 정규화
#     layers.RandomFlip("horizontal"),
#     layers.RandomBrightness(0.1),
#     # 필요시 RandomTranslation 등 추가
# ])

In [ ]:
# # 3. EfficientNet-B0 모델 구축
# def build_model():
#     # ImageNet 가중치를 사용해 전이 학습 (Transfer Learning)
#     # include_top=False로 마지막 분류 레이어는 제외하고 가져옴
#     base_model = applications.EfficientNetB0(
#         include_top=False,
#         weights='imagenet',
#         input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
#     )

#     base_model.trainable = True

#     # 기존 모델의 가중치는 일단 동결 (학습 속도 및 안정성 위해)
#     base_model.trainable = False

#     # 모델 구조 정의
#     inputs = tf.keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
#     x = data_augmentation(inputs) # 전처리 레이어 포함
#     x = base_model(x, training=False)
#     x = layers.GlobalAveragePooling2D()(x) # 특징 맵을 1차원 벡터로 변환
#     x = layers.Dropout(0.2)(x) # 과적합 방지 (보고서 언급)
#     outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x) # 3개 클래스 확률 출력

#     model = tf.keras.Model(inputs, outputs, name="StraightUp_EfficientNetB0")
#     return model

In [ ]:
# # 1. 모델 생성 (이전에 정의한 함수 사용)
# # 만약 함수 정의 셀을 다시 실행해야 한다면 이전 대화의 코드를 참고해줘.
# model = build_model()

# # 2. 모델 컴파일
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
#     loss='sparse_categorical_crossentropy',
#     metrics=['accuracy']
# )

# # 3. 콜백 설정 (학습 효율화)
# # - EarlyStopping: 검증 손실(val_loss)이 5번 연속 안 줄어들면 학습 조기 종료
# early_stopping = tf.keras.callbacks.EarlyStopping(
#     monitor='val_loss',
#     patience=5,
#     restore_best_weights=True
# )

# # 4. 학습 시작 (Epochs는 넉넉하게 20~30 정도 줌, EarlyStopping이 있으니 안심)
# print("학습 시작...")
# history = model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=30,
#     callbacks=[early_stopping]
# )

# # 5. 결과 시각화 (학습 곡선 확인)
# acc = history.history['accuracy']
# val_acc = history.history['val_accuracy']
# loss = history.history['loss']
# val_loss = history.history['val_loss']

# plt.figure(figsize=(8, 8))
# plt.subplot(2, 1, 1)
# plt.plot(acc, label='Training Accuracy')
# plt.plot(val_acc, label='Validation Accuracy')
# plt.legend(loc='lower right')
# plt.ylabel('Accuracy')
# plt.title('Training and Validation Accuracy')

# plt.subplot(2, 1, 2)
# plt.plot(loss, label='Training Loss')
# plt.plot(val_loss, label='Validation Loss')
# plt.legend(loc='upper right')
# plt.ylabel('Cross Entropy')
# plt.title('Training and Validation Loss')
# plt.xlabel('epoch')
# plt.show()

In [ ]:
import tensorflow as tf
import pathlib
from tensorflow.keras.applications.efficientnet import preprocess_input # 치트키!

# 데이터 경로 설정
dataset_path = '/content/drive/MyDrive/Augmented_Dataset'
data_dir = pathlib.Path(dataset_path)

IMG_HEIGHT = 480
IMG_WIDTH = 640
BATCH_SIZE = 16 # 데이터가 적으니 배치 사이즈를 32 -> 16으로 줄이자

# 1. 데이터셋 로드
train_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(IMG_HEIGHT, IMG_WIDTH),
  batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(IMG_HEIGHT, IMG_WIDTH),
  batch_size=BATCH_SIZE
)

# 2. [핵심] EfficientNet 전용 전처리 함수 적용
# 이미지를 EfficientNet이 좋아하는 형태로 자동 변환해 줌
def preprocess(images, labels):
    return preprocess_input(images), labels

train_ds = train_ds.map(preprocess)
val_ds = val_ds.map(preprocess)

# 3. 데이터 캐싱
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("데이터 준비 완료. 전용 전처리 함수가 적용되었습니다.")

In [ ]:
from tensorflow.keras import layers, models, applications

def build_model():
    # Base Model 로드
    base_model = applications.EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
    )

    # [중요] 처음에는 베이스 모델을 학습되지 않도록 '동결(Freeze)' 합니다.
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))

    # Augmentation은 여기서 가볍게만 (이미 증강된 데이터라면 생략 가능하지만, 안전하게 둠)
    x = layers.RandomFlip("horizontal")(inputs)

    # Base Model 통과 (training=False로 설정하여 BatchNormalization 통계 유지)
    x = base_model(x, training=False)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(3, activation='softmax')(x) # 클래스 3개

    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()

# 컴파일 (학습률을 다시 0.001로 올림 - 분류기만 학습하니까)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
print("1차 학습 시작 (분류기만 학습)...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

In [ ]:
# --------------------------------------------------------------
# 1단계 학습을 중단하고 여기서부터 바로 실행하세요!
# --------------------------------------------------------------

import tensorflow as tf

# 1. 잠겨있던 베이스 모델(몸통) 풀기
# 모델 구조에 따라 인덱스가 다를 수 있으니 안전하게 이름으로 찾자
base_model = model.get_layer('efficientnetb0')
base_model.trainable = True

print("🔓 베이스 모델 잠금 해제 완료! 이제 모델 전체가 학습됩니다.")

# 2. [매우 중요] 학습률(Learning Rate)을 아주 낮게 설정
# 몸통의 지식을 유지하면서 미세하게만 튜닝해야 하므로 1e-5 (0.00001) 필수!
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 3. 2차 학습 (Fine-Tuning) 시작
# Epoch는 10~15 정도 주면 충분해.
print("🚀 2차 학습(Fine-Tuning) 시작...")

# history_fine 변수에 결과를 저장해서 그래프를 이어서 그릴 수 있게 함
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

In [ ]:
# 1. TFLite 변환기 생성
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# 2. 최적화 옵션 (모델 크기 경량화)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# 3. 변환 수행
tflite_model = converter.convert()

# 4. 파일로 저장
tflite_path = '/content/straightup_model.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f"TFLite 모델 변환 완료: {tflite_path}")

# 5. 로컬 PC로 다운로드 (Colab 기능)
from google.colab import files
files.download(tflite_path)

print("다운로드가 시작되었습니다. (라벨 순서를 꼭 기억하세요!)")

In [ ]:
# ==========================================
# [Straight Up] EfficientNet-B0 학습 & 변환 통합 스크립트
# ==========================================

import tensorflow as tf
from tensorflow.keras import layers, models, applications
from tensorflow.keras.applications.efficientnet import preprocess_input
import pathlib
import matplotlib.pyplot as plt
import os

# 1. 설정 및 데이터 경로 (구글 드라이브 경로 확인 필수!)
# ------------------------------------------------------
dataset_path = '/content/drive/MyDrive/Augmented_Dataset'  # 증강된 데이터 경로
data_dir = pathlib.Path(dataset_path)

IMG_HEIGHT = 480
IMG_WIDTH = 640
BATCH_SIZE = 16  # 데이터가 적으므로 배치 사이즈 16 권장

print(f"📂 데이터 경로: {dataset_path}")
print("데이터 로드 및 전처리 준비 중...")

# 2. 데이터셋 로드 (Train / Validation)
# ------------------------------------------------------
train_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(IMG_HEIGHT, IMG_WIDTH),
  batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(IMG_HEIGHT, IMG_WIDTH),
  batch_size=BATCH_SIZE
)

# 클래스 순서 확인 (매우 중요)
class_names = train_ds.class_names
print(f"\n[중요] 라벨 순서 (0, 1, 2): {class_names}")

# [핵심] EfficientNet 전용 전처리 함수 적용
def preprocess(images, labels):
    return preprocess_input(images), labels

train_ds = train_ds.map(preprocess)
val_ds = val_ds.map(preprocess)

# 데이터 캐싱 (속도 향상)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)


# 3. 모델 구축 (Base Model 동결)
# ------------------------------------------------------
def build_model():
    base_model = applications.EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
    )

    # 처음엔 베이스 모델 동결 (Freeze)
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
    x = layers.RandomFlip("horizontal")(inputs) # 간단한 증강 추가
    x = base_model(x, training=False) # BatchNormalization 통계 유지
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(3, activation='softmax')(x)

    return tf.keras.Model(inputs, outputs)

model = build_model()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 4. 1차 학습 (분류기만 빠르게 학습)
# ------------------------------------------------------
print("\n🚀 1차 학습 시작 (Head Training)...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=7  # 7 epoch 정도면 충분 (시간 절약)
)

# 5. 2차 학습 (Fine-Tuning: 전체 미세 조정)
# ------------------------------------------------------
print("\n🔓 베이스 모델 잠금 해제 & 미세 조정 시작...")

# 베이스 모델 풀기
base_model = model.get_layer('efficientnetb0')
base_model.trainable = True

# 학습률 아주 낮게 재설정 (필수!)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 이어서 학습
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10, # 추가 10 epoch
    initial_epoch=history.epoch[-1]
)

# 6. TFLite 변환 및 다운로드
# ------------------------------------------------------
print("\n📦 TFLite 변환 중...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = '/content/straightup_model.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f"✅ 변환 완료! 파일 위치: {tflite_path}")

# 로컬 다운로드
from google.colab import files
files.download(tflite_path)
print("⬇️ 다운로드가 시작되었습니다.")

📂 데이터 경로: /content/drive/MyDrive/Augmented_Dataset
데이터 로드 및 전처리 준비 중...
Found 150 files belonging to 3 classes.
Using 120 files for training.
Found 150 files belonging to 3 classes.
Using 30 files for validation.

[중요] 라벨 순서 (0, 1, 2): ['Mild', 'Normal', 'Severe']
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

🚀 1차 학습 시작 (Head Training)...
Epoch 1/7
8/8 ━━━━━━━━━━━━━━━━━━━━ 107s 10s/step - accuracy: 0.4890 - loss: 1.0338 - val_accuracy: 0.4667 - val_loss: 0.9392
Epoch 2/7
8/8 ━━━━━━━━━━━━━━━━━━━━ 76s 10s/step - accuracy: 0.5090 - loss: 0.8700 - val_accuracy: 0.5333 - val_loss: 0.8140
Epoch 3/7
8/8 ━━━━━━━━━━━━━━━━━━━━ 76s 10s/step - accuracy: 0.6991 - loss: 0.6879 - val_accuracy: 0.5667 - val_loss: 0.7711
Epoch 4/7
8/8 ━━━━━━━━━━━━━━━━━━━━ 81s 10s/step - accuracy: 0.7247 - loss: 0.6584 - val_accuracy: 0.6333 - val_loss: 0.7159
Epoch 5/7
8/8 ━━━━━━━━━━━━━━━━━━━━ 82s 10s/step - accuracy: 0.7626 - loss: 0.5935 - val_accuracy: 0.6333 - val_loss: 0.6359
Epoch 6/7
8/8 ━━━━━━━━━━━━━━━━━━

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️ 다운로드가 시작되었습니다.
